<a href="https://colab.research.google.com/github/fwilhelmy/QB-PIER/blob/main/src/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab (Skip locally)

In [ ]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@title Link your assignment folder & install requirements
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys
import os
import shutil
import warnings

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
  warnings.warn('CUDA is not available.')

In [13]:
#@title 📦 Upload ZIP, Extract & Link Your Code
from google.colab import files
import sys, os
import shutil
import warnings

# 1️⃣ Upload your ZIP file
uploaded = files.upload()  # select your .zip
if not uploaded:
    raise FileNotFoundError("No ZIP file uploaded.")
zip_name = list(uploaded.keys())[0]
print(f"✔ Uploaded: {zip_name}")

# 2️⃣ Extract into a folder named after the ZIP (without .zip)
extract_folder = zip_name.rsplit(".", 1)[0]
if os.path.isdir(extract_folder):
    shutil.rmtree(extract_folder)
!unzip -q "{zip_name}" -d "{extract_folder}"
print(f"✔ Extracted to: {extract_folder}/")

# 3️⃣ Symlink that folder to /content/src for easy imports
link_name = "qb-pier"
if os.path.islink(link_name) or os.path.isdir(link_name):
    shutil.rmtree(link_name)
os.symlink(extract_folder, link_name)
print(f"✔ Linked → /content/{link_name}")

# 4️⃣ Add to Python path
if link_name not in sys.path:
    sys.path.insert(0, "{link_name}/src")
    print(f"✔ Added `{link_name}` to sys.path")

# 6️⃣ (Optional) Check for CUDA availability
try:
    import torch
    if not torch.cuda.is_available():
        warnings.warn("CUDA is not available.")
except ImportError:
    pass


FileNotFoundError: No ZIP file uploaded.

# Imports & Setup

In [12]:
%matplotlib inline
import warnings

from main import train_model, run_attacks
from shutil import make_archive


from metrics import LoggingService
from metrics import MetricsContainer
from models import BaseModel
from models.hybrid import HQNN, AmpVQC
from arguments import Arguments

from visualisation.TrainingVisualisation import generate_gradient_flow_plots, plot_training_curves, plot_models_together

ModuleNotFoundError: No module named 'main'

In [ ]:
model_name = "HQNN"  #@param {type:"string"}
model_cls = AmpVQC  #@param {type:"string"}
#path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@14-35-01\\{model_name}\\{model_name}_1\\0"
path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@19-00-06\\{model_name}\\{model_name}_1\\0\\final.ckpt"
path = "C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-30@14-43-55\\AmpVQC\\AmpVQC_1\\0\\final.ckpt"

# load the model
model = model_cls.load_from_checkpoint(path, map_location='cpu')

In [ ]:
#plot_training_result(cont, 42, 42)
#generate_gradient_flow_plots(cont)

# Experiment #1

## Experiment Setup

In [ ]:
# Define a arguments and set global defaults
LoggingService("test_models")
seeds = [0, 42]
args = Arguments()
args.verbose = True
args.n_steps = 2000
args.num_workers = 0

if getattr(args, "num_threads", None) is not None:
    torch.set_num_threads(args.num_threads)

args.eval_ratio = 0.4
args.train_ratio = 0.6

args.train_batch_size = 2**8
args.eval_batch_size = 2**11

args.train_log_interval = 0.5
args.eval_log_interval = 0.5

# 2) Define your list of model‐configurations: (name, class, kwargs)
model_specs = [
    # ("ResNet18_1", ResNet18, {
    #     "input_size": (1, 28),
    #     "num_classes": 10,
    # }),
    # ("MLPMixer_1", MLPMixer, {
    #     "num_classes": 10,
    #     "input_size": (1, 28),
    #     "patch_size": 4,
    #     "embed_dim": 256,
    #     "num_blocks": 4,
    #     "drop_rate": 0.0,
    #     "activation": "gelu",
    #     "mlp_ratio": (0.5, 4.0),
    # }),
    # ("MLP_1", MLP, {
    #     "input_size": (1, 28),
    #     "hidden_sizes": [1024, 512, 64, 64],
    #     "num_classes": 10,
    #     "activation": "relu",
    # }),
    # ("ConvNet_1", ConvNet, {
    #     "input_size": (1, 28),
    #     "num_classes": 10,
    # }),
    # ("LeNet_1", LeNet, {
    #     "in_channels": 1,
    #     "classes": 10,
    # }),
    # ("HQNN_Parallel_1", HQNN, {
    #     "in_channels": 1,
    #     "classes": 10,
    # }),
    # ("HQuanv_1", HQuanv, {
    #     "in_channels": 1,
    #     "classes": 10,
    #     "img_size": 28,
    # }),
    # ("HQE_1", HQE, {
    #     "input_size": 28 * 28,
    #     "hidden_sizes": [256],
    #     "classes": 10,
    # }),
    # ("DimVQC_1", DimVQC, {
    #     "input_size": 28 * 28,
    #     "reduction_method": "pca",
    #     "reduced_dim": 4,
    #     "n_qdepth": 1,
    # }),
    ("AmpVQC_depth1", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 1,
    }),
    ("AmpVQC_depth2", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 2,
    }),
    ("AmpVQC_depth3", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 3,
    }),
    ("AmpVQC_depth4", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 4,
    }),
]

## Models Training

In [ ]:
trained_models = {}
for (model_name, model_cls, config) in model_specs:
    args.exp_name = model_name
    args.model = model_cls
    args.model_config = config

    trained_models[model_name] = train_model(args, seeds)
    plot_training_curves(model_name, trained_models[model_name])
    generate_gradient_flow_plots(list(trained_models[model_name].values())[0])

plot_models_together(trained_models)

## Attacks

In [ ]:
# Run transfer attacks
# attacks_metrics = run_attacks(trained_models, args)
# plot_transfer_attack_effects(attacks_metrics,
#                             title_prefix="transfer")
# from pprint import pprint
# pprint(trained_models['HQNN_Parallel_1'][0].all_metrics, depth=2)
# metrics = trained_models['HQNN_Parallel_1'][0].all_metrics

## Final

In [ ]:
make_archive('experiment1', 'zip', 'logs/experiment1')